# Lab 3 — Prompt Engineering


In [2]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.7 MB/s eta 0:00:00


In [3]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

In [4]:
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [5]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
    )

In [6]:
response = llm.invoke("helloooooo")

In [7]:
print(response.content)

Hello! 👋 How can I help you today?


In [8]:
print(response.response_metadata)

{'token_usage': {'completion_tokens': 46, 'prompt_tokens': 74, 'total_tokens': 120, 'completion_time': 0.048764702, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.004915598, 'prompt_tokens_details': None, 'queue_time': 0.086237477, 'total_time': 0.0536803}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d6de37e6be', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}


Exercise 1: Doctor Summary Generator


In [9]:
from langchain_core.prompts import PromptTemplate

In [10]:
doctor_template = PromptTemplate.from_template("""You are a helpful medical assistant.
Patient Name: {patient_name}
Patient Notes: {patient_notes}

Create a docker's summary in exactly this format:

Symptoms:
diagnosis:
Recommendation:
""")

In [11]:
input_prompt = doctor_template.invoke({
    "patient_name":"Rahul",
    "patient_notes":"Patient has fever and headache for three days. Doctor suspects viral fever and advised rest and hydration."
})

In [12]:
print(input_prompt)

text="You are a helpful medical assistant.\nPatient Name: Rahul\nPatient Notes: Patient has fever and headache for three days. Doctor suspects viral fever and advised rest and hydration. \n\nCreate a docker's summary in exactly this format:\n\nSymptoms:\ndiagnosis:\nRecommendation:\n"


In [13]:
response = llm.invoke(input_prompt)

In [14]:
print(response.content)

Symptoms: Fever and headache for three days
diagnosis: Viral fever
Recommendation: Rest and hydration


Exercise 2 — Review Classifier

In [15]:
review_template = PromptTemplate.from_template(
    """
    Classify the following customer review.
    Categories:
    Positive - Customer is happy or satisfied.
    Negative - Customer is Unhappy or dissatisfied.
    Neutral - Customer is neither happy nor unhappy.
    Customer Review: {review_text}
    Return only one category label.
    """

)

In [16]:
review_input = review_template.invoke({
    "review_text":"The product is amazing and I am very happy with the quality."
})

In [17]:
print(review_input)

text='\n    Classify the following customer review.\n    Categories:\n    Positive - Customer is happy or satisfied.\n    Negative - Customer is Unhappy or dissatisfied.\n    Neutral - Customer is neither happy nor unhappy.\n    Customer Review: The product is amazing and I am very happy with the quality.\n    Return only one category label.\n    '


In [18]:
review_response = llm.invoke(review_input)

In [19]:
print(review_response.content)

Positive


In [21]:
review_input2 = review_template.invoke({
    "review_text":"The product stopped working after two days and im very disappointed."
})

In [22]:
review_response2 = llm.invoke(review_input2)

In [23]:
print(review_response2.content)

Negative


In [38]:
review_input3 = review_template.invoke({
    "review_text": "The product i recieved is okay not so good not so bad  "
})

In [40]:
review_response3 = llm.invoke(review_input3)

In [41]:
print(review_response3.content)

Neutral


Exercise 3 — Resume Field Extractor

In [42]:
from langchain_core.output_parsers import JsonOutputParser

In [44]:
json_parser = JsonOutputParser()

In [47]:
print(json_parser.get_format_instructions())

Return a JSON object.


In [48]:
resume_template = PromptTemplate.from_template(
    """
    Extract only the requested fields from the resume.
    Resume Text:
    {resume_text}
    Fiels to Extract:
    {fields_to_extract}
    Do not include any fields that are not mentioned.

    {JSON_FORMATTING}
    """
).partial(JSON_FORMATTING=json_parser.get_format_instructions())

In [65]:
resume_input = resume_template.invoke({
    "resume_text":"Saniya Shaikh is an MCA student. Her skilss are python, SQL and Machine learning. Her CGPA is 9.2",
    "fields_to_extract": "name, skills"
})

In [67]:
print(resume_input)

text='\n    Extract only the requested fields from the resume.\n    Resume Text:\n    Saniya Shaikh is an MCA student. Her skilss are python, SQL and Machine learning. Her CGPA is 9.2\n    Fiels to Extract:\n    name, skills\n    Do not include any fields that are not mentioned.\n\n    Return a JSON object.\n    '


In [70]:
resume_response = llm.invoke(resume_input)

In [73]:
print(resume_response.content)

```json
{
  "name": "Saniya Shaikh",
  "skills": ["python", "SQL", "Machine learning"]
}
```


In [74]:
final_resume_output = json_parser.invoke(resume_response)

In [75]:
print(final_resume_output)

{'name': 'Saniya Shaikh', 'skills': ['python', 'SQL', 'Machine learning']}


In [77]:
resume_input2 = resume_template.invoke({
    "resume_text": "Saniya Shaikh is an MCA student. Her skilss are python, SQL and Machine learning. Her CGPA is 9.2",
    "fields_to_extract":"name,Education,Cgpa"
})

In [78]:
resume_response2 = llm.invoke(resume_input2)

In [79]:
print(resume_response2.content)

```json
{
  "name": "Saniya Shaikh",
  "Education": "MCA",
  "Cgpa": 9.2
}
```


Exercise 4 — Customer Support Reply Generator

In [80]:
support_template = PromptTemplate.from_template(
    """
    You are a polite and professional customer support assistant.
    Company Name: {company_name}
    Customer Message: {customer_message}

    Write a helpful customer support reply.
    Keep the reply within {max_words} words.
    Maintain a polite and professional tone.
    """
)

In [81]:
support_input = support_template.invoke({
    "company_name":"TechNova",
    "customer_message": "I ordered a wireless mouse five days ago but it still has not arrived.",
    "max_words": 50
})

In [82]:
print(support_input)

text='\n    You are a polite and professional customer support assistant.\n    Company Name: TechNova\n    Customer Message: I ordered a wireless mouse five days ago but it still has not arrived.\n\n    Write a helpful customer support reply.\n    Keep the reply within 50 words.\n    Maintain a polite and professional tone.\n    '


In [83]:
support_response = llm.invoke(support_input)

In [84]:
print(support_response.content)

Dear Customer Name,

I’m sorry your wireless mouse hasn’t arrived. Please share your order number so I can check the shipment status. We’ll resolve this promptly. If you prefer, we can also offer a refund or replacement.

Thank you for your patience.

– TechNova Support


In [85]:
word_count = len(support_response.content.split())
print("Word Count:", word_count)

Word Count: 45


Exercise 5 — Multi-language Translator

In [86]:
translate_template = PromptTemplate.from_template(
    """
    Translate the following sentence.
    sentence: {sentence}
    Target Language: {target_language}
    Formality Level: {formality}

    Return only the translated sentence.
    """
)

In [94]:
translate_input = translate_template.invoke({
    "sentence": "Can you help me with this work?",
    "target_language":"Urdu",
    "formality":"Formal"
})

In [96]:
print(translate_input)

text='\n    Translate the following sentence.\n    sentence: Can you help me with this work?\n    Target Language: Urdu\n    Formality Level: Formal\n\n    Return only the translated sentence.\n    '


In [98]:
translate_response = llm.invoke(translate_input)

In [99]:
print(translate_response.content)

کیا آپ مجھے اس کام میں مدد کر سکتے ہیں؟


In [104]:
translate_input2 = translate_template.invoke({
    "sentence": "Hi My name is saniya shaikh",
    "target_language": "Urdu",
    "formality": "Informal"
})

In [105]:
translate_response2 = llm.invoke(translate_input2)

In [106]:
print(translate_response2.content)

ہیلو، میرا نام سانیہ شیک ہے


Exercise 6 — Iterative Content Refiner

In [107]:
refiner_template = PromptTemplate.from_template(
    """
    Improve the following content.
    Draft Text:
    {draft_text}

    Issue to Fix:
    {issue_to_fix}

    Return only the revised text.
    """
)

In [108]:
draft = "AI is good. AI helps people. AI is used in many places. AI is useful."

In [109]:
issue = "Remove repetition and improve sentence flow."

In [110]:
refiner_input = refiner_template.invoke({
    "draft_text": draft,
    "issue_to_fix":issue
})

In [111]:
print(refiner_input)

text='\n    Improve the following content.\n    Draft Text:\n    AI is good. AI helps people. AI is used in many places. AI is useful.\n\n    Issue to Fix:\n    Remove repetition and improve sentence flow.\n\n    Return only the revised text.\n    '


In [112]:
round1_response = llm.invoke(refiner_input)

In [113]:
round1 = round1_response.content

In [114]:
print(round1)

AI is beneficial, aiding people across many domains and proving its usefulness in diverse applications.


In [115]:
refiner_input2 = refiner_template.invoke({
    "draft_text": round1,
    "issue_to_fix": issue

})

In [117]:
round2_response = llm.invoke(refiner_input2)

In [118]:
round2 = round2_response.content

In [119]:
print(round2)

AI benefits people across many domains, proving its value in a wide array of applications.


In [120]:
refiner_input3 = refiner_template.invoke({
    "draft_text": round2,
    "issue_to_fix": issue
})

In [125]:
round3_response = llm.invoke(refiner_input3)
round3 = round3_response.content
print(round3)

AI benefits people across diverse domains, illustrating its value in countless applications.
